In [11]:
import pandas as pd
import numpy as np
import os
import json
import requests
from pyjstat import pyjstat
from collections import OrderedDict

Eurostat queries based on query builder: https://ec.europa.eu/eurostat/web/json-and-unicode-web-services/getting-started/query-builder

In [12]:
dir_in = "../source_data/Eurostat/"
dir_out = "../parsed_data/"

In [13]:
map_country_ISO = {
    "Belgium" : "BE",
    "Bulgaria" : "BG",
    "Czechia" : "CZ",
    "Denmark" : "DK",
    "Germany" : "DE",  # Leon edit, just Germany
    "Estonia" : "EE",
    "Ireland" : "IE",
    "Greece" : "EL",  # Leon edit, EL = Griechenland nach eurostat 
    "Spain" : "ES",
    "France" : "FR",
    "Croatia" : "HR",
    "Italy" : "IT",
    "Cyprus" : "CY",      # Leon edit, added Cyprus 
    "Latvia" : "LV",
    "Lithuania" : "LT",
    "Luxembourg" : "LU",
    "Hungary" : "HU",
    "Malta" : "MT",       # Leon edit, added Malta
    "Netherlands" : "NL",
    "Austria" : "AT",
    "Poland" : "PL" ,
    "Portugal" : "PT",
    "Romania" : "RO",
    "Slovenia" : "SI",
    "Slovakia" : "SK",
    "Finland" : "FI",
    "Sweden" : "SE",
    "Iceland" : "IS",      # Leon edit, added Iceland
    "United Kingdom" : "UK",  # Leon edit, UK nach eurostat 
    "Norway" : "NO",
    "Bosnia and Herzegovina" : "BA",   # Leon edit, added: 
    "Montenegro" : "ME", 
    "Moldova" : "MD",
    "North Macedonia" : "MK",
    "Georgia" : "GE",
    "Albania" : "AL",
    "Serbia" : "RS",
    "Türkiye" : "TR",
    "Ukraine" : "UA",
    "Kosovo*" : "XK",
    "Liechtenstein" : "LI"      ##### LI ist das einzige angefragte, aber nicht von eurostat verfügbare Land ##### 
}
print(list(map_country_ISO.values()))

['BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV', 'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 'PL', 'PT', 'RO', 'SI', 'SK', 'FI', 'SE', 'IS', 'UK', 'NO', 'BA', 'ME', 'MD', 'MK', 'GE', 'AL', 'RS', 'TR', 'UA', 'XK', 'LI']


In [14]:
# Leon edit, neue Namen und Kategorien aus eurostat
map_tech = {
    'Total' : 'Total',
    'Hydro' : 'Hydro', # Hydro-Aggregat (= RA100 = RA110 + RA120 + RA130(= pump)) 
    'Pumped hydro power' : 'Pump', # Hydro-Teilmenge (= RA130)
    'Geothermal' : 'Other',
    'Nuclear fuels and other fuels n.e.c.' : 'Nuclear', # einzige Kategorie mit Nuclear
    'Coal and manufactured gases' : 'Coal',
    'Oil and petroleum products (excluding biofuel portion)' : 'Oil',
    'Natural gas' : 'Gas',
    'Combustible fuels - renewable' : 'Biomass',
    'Combustible fuels - non-renewable' : 'Other' ,
    'Wind' : 'Wind',
    'Wind on shore' : 'WindOnshore',
    'Wind off shore' : 'WindOffshore',
    'Solar' : 'Solar',  # Solar-Aggregat (= RA400 = RA410 + RA420) 
    'Other renewable energies' : 'Other',
    'Other fuels n.e.c.' : 'Other',
    
}

load nrg_105_m which has detailed monthly generation data per technology and country in GWh

In [15]:
indicator = 'nrg_cb_pem'  # Leon edit, neue URL für neuen Datensatz
dataformat = 'JSON'   

# Leon edit: 
params = dict(
    sinceTimePeriod = '2017-01',    
    geo = {'AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MD', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR', 'UA', 'UK', 'AL', 'BA', 'LI', 'IS', 'GE', 'ME', 'XK'},
    unit = 'GWH',
    # Standard international energy product classification (SIEC) - alle Verfügbaren von eurostat angefragt
    siec = {'C0000', 'CF', 'CF_NR', 'CF_R', 'FE', 'G3000', 'N9000', 'O4000XBIO', 'RA000', 'RA100', 'RA110', 'RA120', 'RA130', 'RA200', 'RA300', 'RA310', 'RA320', 'RA400', 'RA410', 'RA420', 'RA500_5160', 'TOTAL', 'X9900'},
)

print(", ".join(params["geo"]))

TR, SK, EE, IS, PL, XK, UA, BE, IE, MT, GE, PT, SI, LU, HR, FI, LI, BA, CZ, ES, SE, FR, HU, AT, ME, MD, AL, EL, BG, LV, CY, MK, RS, LT, RO, DK, NL, IT, UK, NO, DE


In [16]:
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"+indicator+"?format="+dataformat   # Leon edit, updated URL
r = requests.get(url=url, params=params)

print(r.url)

https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nrg_cb_pem?format=JSON&sinceTimePeriod=2017-01&geo=TR&geo=SK&geo=EE&geo=IS&geo=PL&geo=XK&geo=UA&geo=BE&geo=IE&geo=MT&geo=GE&geo=PT&geo=SI&geo=LU&geo=HR&geo=FI&geo=LI&geo=BA&geo=CZ&geo=ES&geo=SE&geo=FR&geo=HU&geo=AT&geo=ME&geo=MD&geo=AL&geo=EL&geo=BG&geo=LV&geo=CY&geo=MK&geo=RS&geo=LT&geo=RO&geo=DK&geo=NL&geo=IT&geo=UK&geo=NO&geo=DE&unit=GWH&siec=RA000&siec=RA310&siec=RA300&siec=RA400&siec=RA420&siec=RA110&siec=RA320&siec=N9000&siec=RA410&siec=O4000XBIO&siec=G3000&siec=RA120&siec=CF_R&siec=C0000&siec=CF&siec=FE&siec=CF_NR&siec=RA200&siec=RA130&siec=RA500_5160&siec=RA100&siec=X9900&siec=TOTAL


In [17]:
# Rohdaten Data Frame: 
df_nrg_cb_pem = pd.DataFrame(pyjstat.from_json_stat(r.json(object_pairs_hook=OrderedDict))[0])
df_nrg_cb_pem.tail(70)


,Time frequency,Standard international energy product classification (SIEC),Unit of measure,Geopolitical entity (reporting),Time,value
100210,Monthly,Fossil energy,Gigawatt-hour,Kosovo*,2020-04,NaN
100211,Monthly,Fossil energy,Gigawatt-hour,Kosovo*,2020-05,NaN
100212,Monthly,Fossil energy,Gigawatt-hour,Kosovo*,2020-06,NaN
100213,Monthly,Fossil energy,Gigawatt-hour,Kosovo*,2020-07,NaN
100214,Monthly,Fossil energy,Gigawatt-hour,Kosovo*,2020-08,NaN
...,...,...,...,...,...,...
100275,Monthly,Fossil energy,Gigawatt-hour,Kosovo*,2025-09,375.995
100276,Monthly,Fossil energy,Gigawatt-hour,Kosovo*,2025-10,503.990
100277,Monthly,Fossil energy,Gigawatt-hour,Kosovo*,2025-11,495.231
100278,Monthly,Fossil energy,Gigawatt-hour,Kosovo*,2025-12,592.300


In [18]:
# Ausgabe aller SIEC's (Technologien) in dem Rohdaten - Data Frame (in der Reihenfolge wie sie von oben nach unten im Data Frame vorkommen)
df_nrg_cb_pem['Standard international energy product classification (SIEC)'].unique()

array(['Total', 'Combustible fuels', 'Combustible fuels - renewable',
       'Combustible fuels - non-renewable', 'Coal and manufactured gases',
       'Natural gas',
       'Oil and petroleum products (excluding biofuel portion)',
       'Renewables and biofuels', 'Hydro', 'Pure hydro power',
       'Mixed hydro power', 'Pumped hydro power', 'Geothermal', 'Wind',
       'Wind on shore', 'Wind off shore', 'Solar', 'Solar thermal',
       'Solar photovoltaic', 'Other renewable energies',
       'Nuclear fuels and other fuels n.e.c.', 'Other fuels n.e.c.',
       'Fossil energy'], dtype=object)

In [19]:
# Leon:
# Geg. Aussortieren von unbrauchbaren/unerwünschten "Technologie"-Kategorien (=SIEC) aus den Rohdaten: 

# Liste der unbrauchbaren/unerwünschten "Technologien" aus den Rohdaten

# 1. Sub-Kategorien, bei denen wir das Aggregat haben: 
# Pure hydro power, Mixed hydro power, Solar thermal, Solar photovoltaic
# Pumped hydro power nehmen wir als Sub-Kategorie von Hydro auf 
# 2. Oberkategorien, bei denen wir die Subkategorien haben: 
# Combustible fuels, Renewables and biofuels, Fossil energy

exclude_tech = ['Pure hydro power','Mixed hydro power','Solar thermal','Solar photovoltaic','Combustible fuels','Renewables and biofuels','Fossil energy']

# Filter direkt im Rohdaten - Data Frame
df_nrg_cb_pem = df_nrg_cb_pem[~df_nrg_cb_pem['Standard international energy product classification (SIEC)'].isin(exclude_tech)]

# Ausgabe des überschriebenen Data Frames (jetzt ohne nicht erwünschte "Technologien")
#df_nrg_cb_pem.tail()
df_nrg_cb_pem['Standard international energy product classification (SIEC)'].unique()

array(['Total', 'Combustible fuels - renewable',
       'Combustible fuels - non-renewable', 'Coal and manufactured gases',
       'Natural gas',
       'Oil and petroleum products (excluding biofuel portion)', 'Hydro',
       'Pumped hydro power', 'Geothermal', 'Wind', 'Wind on shore',
       'Wind off shore', 'Solar', 'Other renewable energies',
       'Nuclear fuels and other fuels n.e.c.', 'Other fuels n.e.c.'],
      dtype=object)

In [20]:
#df_nrg_cb_pem = pd.DataFrame(pyjstat.from_json_stat(r.json(object_pairs_hook=OrderedDict))[0])
df_nrg_cb_pem = df_nrg_cb_pem.rename(columns={"Time": "time", "Geopolitical entity (reporting)": "country",
                                      "Standard international energy product classification (SIEC)": "tech", "value": "value",
                                       "Unit of measure":"unit"})
df_nrg_cb_pem['country'] = df_nrg_cb_pem['country'].map(map_country_ISO)
df_nrg_cb_pem['tech'] = df_nrg_cb_pem['tech'].map(map_tech)
df_nrg_cb_pem['MWh'] = df_nrg_cb_pem['value'] * 1000
df_nrg_cb_pem = df_nrg_cb_pem.drop(columns = ['Time frequency','unit','value']) # Leon, added "Time frequency"
df_nrg_cb_pem = df_nrg_cb_pem.groupby(['tech','country','time']).sum()
df_nrg_cb_pem.tail()

# Leon:
#print(set(params["geo"]) - set(df_nrg_cb_pem["country"]))

#df_nrg_cb_pem['country'].unique()

#df_nrg_cb_pem.groupby(['time', 'tech'])['value'].sum().unstack().fillna(0)

#df_nrg_cb_pem['tech'].unique()

MWh
tech        country time            
WindOnshore XK      2025-09  17679.0
                    2025-10  38726.0
                    2025-11  34640.0
                    2025-12  29954.0
                    2026-01      0.0

In [21]:
# Nuller - Länder: 

# Summe pro Land über alle Zeiträume berechnen
country_sums = df_nrg_cb_pem.groupby('country')['MWh'].sum()

# Länder herausfiltern, deren gesamte Summe 0 ist
zero_countries = country_sums[country_sums == 0].index.tolist()

print("Länder mit nur 0 über alle Jahre:", zero_countries)
# es gibt keine Nuller-Länder

Länder mit nur 0 über alle Jahre: []


Some country values are only available on an aggregate basis, so we add them manually, where needed

In [22]:
df_nrg_cb_pem_pivot = df_nrg_cb_pem.reset_index().pivot_table(columns='tech',index=['country','time'],values='MWh')
#Estonia only has onshore wind but is just reported as wind (https://en.wikipedia.org/wiki/Wind_power_in_Estonia):
#df_nrg_105m_pivot.WindOnshore[df_nrg_105m_pivot.index.get_level_values('country') == 'EE'] = df_nrg_105m_pivot.Wind[df_nrg_105m_pivot.index.get_level_values('country') == 'EE']
#same for Italy (https://it.wikipedia.org/wiki/Eolico_offshore)
#df_nrg_105m_pivot.WindOnshore[df_nrg_105m_pivot.index.get_level_values('country') == 'IT'] = df_nrg_105m_pivot.Wind[df_nrg_105m_pivot.index.get_level_values('country') == 'IT']

df_nrg_cb_pem_pivot.tail()

tech             Biomass      Coal  Gas    Hydro  Nuclear  Oil  Other  Pump  \
country time                                                                  
XK      2025-09      0.0  375995.0  0.0   3864.0      0.0  0.0    0.0   0.0   
        2025-10      0.0  503990.0  0.0   8846.0      0.0  0.0    0.0   0.0   
        2025-11      0.0  495231.0  0.0  26019.0      0.0  0.0    0.0   0.0   
        2025-12      0.0  592300.0  0.0  34962.0      0.0  0.0    0.0   0.0   
        2026-01      0.0       0.0  0.0      0.0      0.0  0.0    0.0   0.0   

tech              Solar     Total     Wind  WindOffshore  WindOnshore  
country time                                                           
XK      2025-09  1770.0  399308.0  17679.0           0.0      17679.0  
        2025-10  1115.0  552677.0  38726.0           0.0      38726.0  
        2025-11  1200.0  557090.0  34640.0           0.0      34640.0  
        2025-12  1224.0  658440.0  29954.0           0.0      29954.0  
        2026-01     0.0       0.0      0.0           0.0          0.0

In [23]:
# Estonia:
df_nrg_cb_pem_pivot.loc['EE'].iloc[12+12*7:24+12*7]

tech,Biomass,Coal,Gas,Hydro,Nuclear,Oil,Other,Pump,Solar,Total,Wind,WindOffshore,WindOnshore
time,,,,,,,,,,,,,
2025-01,129930.0,0.0,0.0,4100.0,0.0,0.0,199750.0,0.0,7000.0,521080.0,180300.0,0.0,180300.0
2025-02,128644.0,0.0,0.0,3300.0,0.0,0.0,279181.0,0.0,27300.0,517925.0,79500.0,0.0,79500.0
2025-03,125118.0,0.0,0.0,4100.0,0.0,0.0,196973.0,0.0,97300.0,577991.0,154500.0,0.0,154500.0
2025-04,111309.0,0.0,0.0,2800.0,0.0,0.0,189341.0,0.0,136000.0,555150.0,115700.0,0.0,115700.0
2025-05,91802.0,0.0,0.0,2500.0,0.0,0.0,125022.0,0.0,176700.0,491124.0,95100.0,0.0,95100.0
2025-06,53782.0,0.0,0.0,2400.0,0.0,0.0,104010.0,0.0,154300.0,419892.0,105400.0,0.0,105400.0
2025-07,39539.0,0.0,0.0,3000.0,0.0,0.0,36647.0,0.0,179100.0,329186.0,70900.0,0.0,70900.0
2025-08,56432.0,0.0,0.0,1900.0,0.0,0.0,23267.0,0.0,149300.0,296299.0,65400.0,0.0,65400.0
2025-09,64193.0,0.0,0.0,2600.0,0.0,0.0,82974.0,0.0,119800.0,363467.0,93900.0,0.0,93900.0


In [24]:
# Italy: 
df_nrg_cb_pem_pivot.loc['IT'].iloc[12+12*3:24+12*3]

tech,Biomass,Coal,Gas,Hydro,Nuclear,Oil,Other,Pump,Solar,Total,Wind,WindOffshore,WindOnshore
time,,,,,,,,,,,,,
2021-01,0.0,0.0,0.0,3749000.0,0.0,0.0,465000.0,114000.0,914000.0,23904000.0,2604000.0,0.0,2604000.0
2021-02,0.0,0.0,0.0,3532000.0,0.0,0.0,427000.0,146000.0,1467000.0,20655000.0,1697000.0,0.0,1697000.0
2021-03,0.0,0.0,0.0,3190000.0,0.0,0.0,475000.0,129000.0,2415000.0,22395000.0,1826000.0,0.0,1826000.0
2021-04,0.0,0.0,0.0,3182000.0,0.0,0.0,459000.0,125000.0,2425000.0,21315000.0,1541000.0,0.0,1541000.0
2021-05,0.0,0.0,0.0,4666000.0,0.0,0.0,465000.0,146000.0,2998000.0,21226000.0,1969000.0,0.0,1969000.0
2021-06,0.0,0.0,0.0,5683000.0,0.0,0.0,456000.0,93000.0,3003000.0,23839000.0,960000.0,0.0,960000.0
2021-07,0.0,0.0,0.0,5268000.0,0.0,0.0,470000.0,77000.0,2944000.0,26161000.0,1403000.0,0.0,1403000.0
2021-08,0.0,0.0,0.0,4835000.0,0.0,0.0,463000.0,115000.0,2928000.0,23134000.0,1424000.0,0.0,1424000.0
2021-09,0.0,0.0,0.0,3124000.0,0.0,0.0,458000.0,122000.0,2343000.0,23094000.0,986000.0,0.0,986000.0


In [25]:
# UK: 
df_nrg_cb_pem_pivot.loc['UK'].iloc[12+12*1:24+12*1]

tech,Biomass,Coal,Gas,Hydro,Nuclear,Oil,Other,Pump,Solar,Total,Wind,WindOffshore,WindOnshore
time,,,,,,,,,,,,,
2019-01,2611618.0,1819846.0,14863467.0,734012.0,4421661.0,127560.0,473306.0,189071.0,469610.0,31171672.0,5650592.0,2779070.0,2871522.0
2019-02,2254773.0,621619.0,10827894.0,675542.0,4138371.0,81576.0,426983.0,157814.0,722517.0,25676946.0,5927670.0,2576053.0,3351617.0
2019-03,2562397.0,449866.0,10382710.0,881735.0,4070716.0,60354.0,479397.0,162360.0,930248.0,26754369.0,6936947.0,3203838.0,3733108.0
2019-04,2308791.0,297600.0,11071306.0,443064.0,4423333.0,55309.0,431139.0,99973.0,1530847.0,25298574.0,4737185.0,2283721.0,2453463.0
2019-05,2672692.0,39164.0,11103843.0,291956.0,4396399.0,75695.0,457228.0,75151.0,1692789.0,23932446.0,3202679.0,1565029.0,1637650.0
2019-06,2528975.0,93325.0,10464368.0,467881.0,3048893.0,67277.0,459544.0,116162.0,1557770.0,22846822.0,4158791.0,2109925.0,2048866.0
2019-07,2530008.0,128291.0,11278479.0,427799.0,3708124.0,69679.0,461249.0,94943.0,1633413.0,23771903.0,3534861.0,1709118.0,1825742.0
2019-08,2549033.0,333352.0,8286737.0,600063.0,4139790.0,58355.0,468724.0,109976.0,1544795.0,23200063.0,5219216.0,2647450.0,2571765.0
2019-09,2399829.0,227422.0,8615733.0,699015.0,4498382.0,70917.0,438135.0,105267.0,1345968.0,23654003.0,5358602.0,2863499.0,2495103.0


In [26]:
df_nrg_cb_pem = pd.DataFrame(df_nrg_cb_pem_pivot.stack()).rename(columns={0:'MWh'})
df_nrg_cb_pem.tail(20)

MWh
country time    tech                  
XK      2025-12 Other              0.0
                Pump               0.0
                Solar           1224.0
                Total         658440.0
                Wind           29954.0
                WindOffshore       0.0
                WindOnshore    29954.0
        2026-01 Biomass            0.0
                Coal               0.0
                Gas                0.0
                Hydro              0.0
                Nuclear            0.0
                Oil                0.0
                Other              0.0
                Pump               0.0
                Solar              0.0
                Total              0.0
                Wind               0.0
                WindOffshore       0.0
                WindOnshore        0.0

In [27]:
df_nrg_cb_pem['year'] = df_nrg_cb_pem.index.get_level_values('time').str[:4]
df_nrg_cb_pem['month'] = df_nrg_cb_pem.index.get_level_values('time').str[5:7]
df_nrg_cb_pem = df_nrg_cb_pem[['year','month','MWh']]
df_nrg_cb_pem.tail(20)

year month       MWh
country time    tech                              
XK      2025-12 Other         2025    12       0.0
                Pump          2025    12       0.0
                Solar         2025    12    1224.0
                Total         2025    12  658440.0
                Wind          2025    12   29954.0
                WindOffshore  2025    12       0.0
                WindOnshore   2025    12   29954.0
        2026-01 Biomass       2026    01       0.0
                Coal          2026    01       0.0
                Gas           2026    01       0.0
                Hydro         2026    01       0.0
                Nuclear       2026    01       0.0
                Oil           2026    01       0.0
                Other         2026    01       0.0
                Pump          2026    01       0.0
                Solar         2026    01       0.0
                Total         2026    01       0.0
                Wind          2026    01       0.0
                WindOffshore  2026    01       0.0
                WindOnshore   2026    01       0.0

In [28]:
df_nrg_cb_pem_year = df_nrg_cb_pem.groupby(['tech','country','year']).sum().reset_index()

# Leon, Löschen der Monats-Spalte: 

df_nrg_cb_pem_year = df_nrg_cb_pem_year.drop(columns = 'month')

# Ausgabe des Überschriebenen (ohne Monatsspalte): 
df_nrg_cb_pem_year.tail(20)

,tech,country,year,MWh
5180,WindOnshore,UK,2017,30797693.0
5181,WindOnshore,UK,2018,30216899.0
5182,WindOnshore,UK,2019,32205379.0
5183,WindOnshore,UK,2020,28566192.0
5184,WindOnshore,UK,2021,0.0
5185,WindOnshore,UK,2022,0.0
5186,WindOnshore,UK,2023,0.0
5187,WindOnshore,UK,2024,0.0
5188,WindOnshore,UK,2025,0.0
5189,WindOnshore,UK,2026,0.0


In [29]:
#export this to CSV
df_nrg_cb_pem.to_csv(dir_out + 'generation_monthly_eurostat.csv',index=True)
df_nrg_cb_pem_year.to_csv(dir_out + 'generation_yearly_eurostat.csv',index=False)